# Home Credit Default Risk: Modelling Pipeline

**Purpose.** This notebook documents the full modelling workflow of a credit-decision support product: problem definition, data preparation, modelling, evaluation, model selection and the hand-over to deployment. It starts from the raw Home Credit files on Google Drive (the exploratory analysis is delivered separately) and ends with models trained on 100% of the data plus a deployment bundle.
Runs on Google Colab (GPU recommended).

## 1. Problem definition

**Business problem.** A loan officer must decide on every consumer-loan application. Most applications are low risk and need little attention; a minority are high risk and need a closer look. Manual review of everything is slow and inconsistent.

**Solution.** A decision-support service that, for each application:
1. estimates the **probability of default** (a calibrated probability, not only a ranking),
2. routes the application to a **review lane** (fast, standard, strict),
3. checks the **SAMA salary-linked obligation cap** and an age-eligibility rule,
4. **explains** the main factors behind the estimate,
5. (phase 2) suggests a **smaller loan amount** that satisfies the cap and the risk threshold, and gives an **early warning** for running contracts.

The service supports the officer; it never approves or rejects on its own.

**Target.** `TARGET = 1` marks a client with payment difficulties on the loan; the default rate is low, so the problem is **imbalanced binary classification**.

**Constraints that shape the design.**
- Probabilities must be **calibrated**, because lanes, the SAMA check and amount suggestions use the probability value, not only the ranking.
- Gender and age are **not model inputs**. Age is applied as an eligibility rule outside the model; gender and age are used only to audit performance by group.
- Only information **available at application time** may be used (no leakage from later events).

**Success criteria.** Ranking quality (AUC), capture of defaulters (PR-AUC), probability accuracy (Brier score, ECE), stable performance across customer segments, and a model that can be explained and served.

## 2. Workflow and where each stage is covered

| Stage | Where |
|---|---|
| Problem definition | This section |
| EDA | Delivered separately; a compact data overview is computed in Section 1 of this notebook |
| Preprocessing and feature engineering | Section 1 (steps 1 to 4) |
| Modelling | Sections 2 to 5 (algorithm comparison per model) |
| Evaluation | Metrics and interpretation cells in every model section |
| Model selection | Selection rule in steps 6, 12, 14, 17; summary in step 19 |
| Deployment | Final training (steps 11, 13, 16, 18), bundle (step 19), FastAPI service and web UI in the application repository |

## 3. Contents

| Section | Content |
|---|---|
| 0. Configuration and setup | Run settings, dependencies, Google Drive |
| 1. Data preparation | CSV to Parquet, feature engineering, data overview, input rules, SAMA affordability cap |
| 2. Model 1: default risk | Algorithm comparison, calibration, segment audit, confidence, triage lanes, explanations, final training |
| 3. Model 2: fallback (no external scores) | Algorithm comparison, final training |
| 4. Model 3: monotone risk and amount suggestion | Algorithm comparison, amount suggestion, final training |
| 5. Model 4: early warning | Algorithm comparison, final training |
| 6. Summary, decisions and hand-over | Results table, decision log, conclusions, deployment, next steps |

## 4. Model dependencies

| Component | Depends on |
|---|---|
| Model 1 | Feature matrix |
| Calibration, segment audit, confidence, triage lanes, explanations | Model 1 |
| Model 2 | Feature matrix (independent of Model 1) |
| Model 3 | Feature matrix (independent of Model 1) |
| Amount suggestion | Model 3 and the SAMA affordability cap |
| Model 4 | `POS_CASH_balance` (independent of Models 1 to 3) |

## 5. Before running

1. Upload the 8 raw CSV files to `MyDrive/home_credit_raw/`.
2. Select a GPU runtime (Runtime, Change runtime type, T4 GPU).
3. Run all cells in order. Every experiment and every final model is saved to Drive when it finishes, and finished work is loaded from Drive on re-run.

### Step 0. Configuration

| | |
|---|---|
| **Input** | None. |
| **Purpose** | Single place for all run settings: paths, algorithms, evaluation folds, protected attributes, SAMA caps, eligibility ages. |
| **Depends on** | None. |
| **Output** | Settings used by every later step. |

In [ ]:
# STEP 0 - settings. Change values here only.
DRIVE_RAW = "/content/drive/MyDrive/home_credit_raw"     # folder with the 8 raw CSV files
RUN_DIR   = "/content/drive/MyDrive/home_credit_runs"    # every result / model is saved here (survives disconnects)
USE_GPU   = True        # select a T4 GPU runtime first; XGBoost + CatBoost use it
USE_GENDER = False      # put CODE_GENDER inside the risk model?
USE_AGE    = False      # put age (DAYS_BIRTH) inside the risk model? (age can still be an eligibility RULE, step 3)
EVAL_FOLDS = [0, 1, 2, 3, 4]          # 5-fold evaluation on ALL rows; use [0] for a quicker first look
ALGOS      = ["logreg", "lightgbm", "xgboost", "catboost"]   # algorithms compared for models 1, 2 and 4
ALGOS_MONO = ["lightgbm", "xgboost"]  # model 3 needs monotone constraints (CatBoost monotone is CPU-only and very slow)
SAMA_CAP_EMPLOYEE, SAMA_CAP_RETIRED = 1 / 3, 0.25   # SAMA: salary-linked monthly obligations / gross salary
MIN_AGE, MAX_AGE_AT_MATURITY = 21, 65               # example eligibility policy (bank's choice, not a SAMA number)
SEED = 42
RETRAIN_FINAL = False   # False: a final model already saved in Drive is reused (resume after a disconnect). True: retrain all (use after changing any setting above)

### Environment setup

| | |
|---|---|
| **Input** | None. |
| **Purpose** | Installs dependencies, mounts Google Drive and creates the output folders. |
| **Depends on** | Step 0. |
| **Output** | Imports, `RUN` output folder. |

In [ ]:
# SETUP - install libraries, mount Google Drive, create working folders
import os, sys, subprocess, json, time, warnings
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "catboost", "xgboost==2.1.4", "lightgbm"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/work")                         # fast local disk for parquet copies
else:                                                    # local test run (developer only)
    DRIVE_RAW = os.environ.get("HC_RAW", DRIVE_RAW); RUN_DIR = os.environ.get("HC_RUN", RUN_DIR)
    WORK = Path(os.environ.get("HC_WORK", "./_work"))
SMOKE_ROWS = int(os.environ.get("HC_SMOKE_ROWS", "0")) or None   # developer test only; leave unset on Colab
RUN = Path(RUN_DIR); WORK.mkdir(parents=True, exist_ok=True)
for d in [RUN, RUN / "experiments", RUN / "models", RUN / "final"]:
    d.mkdir(parents=True, exist_ok=True)
import duckdb, numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
if IN_COLAB and xgb.__version__ != "2.1.4":          # the app runs XGBoost 2.1.4 (newest version for Python 3.9); models must be trained with it
    raise RuntimeError(f"XGBoost {xgb.__version__} is loaded, 2.1.4 is required. Use Runtime > Restart session, then run all cells again.")
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")
print("raw:", DRIVE_RAW, "| results:", RUN, "| GPU:", USE_GPU)

# Section 1. Data preparation

Converts the raw files, builds the as-of-application features and applies the rule-based checks. No model is trained in this section.

## Preprocessing and feature-engineering decisions

| Decision | Choice | Reason |
|---|---|---|
| Time frame of history features | Aggregate each auxiliary table **only with information dated before the application** (records updated afterwards are excluded) | Prevents target leakage: the model must score a new applicant with what is known at decision time |
| One row per client | Aggregate bureau, previous-application, instalment and credit-card tables to client level | The application table has one row per application; counts, means, maxima, recent-window and trend features keep the behavioural signal |
| Missing values | Left missing for gradient-boosting models; median imputation plus missing-indicator for logistic regression | Tree models learn from missingness natively, and "no history" is itself informative |
| Missing history of an obligation | Treated as no known obligation (0) in affordability ratios | A client with no bureau record has no known bureau debt |
| Sentinel values | `DAYS_EMPLOYED = 365243` converted to missing | It encodes "not applicable", not a real duration |
| Categorical variables | Native categorical handling in LightGBM, XGBoost and CatBoost; one-hot (rare levels merged) for logistic regression; identical category levels in train and test | Avoids arbitrary integer codes and train/test level mismatch |
| Scaling | Only for logistic regression | Tree models are invariant to monotone scaling |
| Affordability features | Instalment, credit and debt relative to income, and residual income per family member | Mirrors how a credit officer and SAMA rules assess repayment capacity |
| Protected attributes | `CODE_GENDER` and `DAYS_BIRTH` excluded from model inputs (switchable in step 0) | Non-discrimination; age handled as an explicit eligibility rule |
| Income period | Treated as monthly | The dataset does not document the period; treating it as annual would make almost every approved loan violate the cap, which is implausible |

### Step 1. Convert raw CSV files to Parquet

| | |
|---|---|
| **Input** | The 8 raw CSV files in `DRIVE_RAW` (read-only). |
| **Purpose** | Typed, columnar copies of every table for fast processing with DuckDB. Raw files are never modified. |
| **Depends on** | Step 0. |
| **Output** | One Parquet file per table in the working folder. |

In [ ]:
# STEP 1 - raw CSV (read-only) -> local parquet; numbers are typed, Y/N flags stay text
TABLES = ["application_train", "application_test", "bureau", "bureau_balance", "previous_application",
          "installments_payments", "credit_card_balance", "POS_CASH_balance"]
con = duckdb.connect()
con.execute("SET memory_limit='6GB'"); con.execute("SET threads=4")
con.execute(f"SET temp_directory='{(WORK / 'duck_tmp').as_posix()}'")
for t in TABLES:
    src, dst = Path(DRIVE_RAW) / f"{t}.csv", WORK / f"{t}.parquet"
    assert src.exists(), f"missing {src}"
    if not dst.exists():
        t0 = time.time()
        con.execute(f"COPY (SELECT * FROM read_csv('{src.as_posix()}', header=true, sample_size=-1, "
                    f"auto_type_candidates=['BIGINT','DOUBLE','VARCHAR'])) TO '{dst.as_posix()}' (FORMAT PARQUET)")
        print(f"{t}: {time.time()-t0:.0f}s")
    con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_parquet('{dst.as_posix()}')")
print(con.execute("SELECT COUNT(*) FROM application_train").fetchone()[0], "training applications")

## Step 2. Feature engineering

Each auxiliary table is aggregated to one row per client using only information available before the application date. Steps 2a to 2e build the feature tables and step 2f assembles them.

### Step 2a. Instalment-payment features

| | |
|---|---|
| **Input** | `installments_payments`. |
| **Purpose** | Per-client payment behaviour: lateness, shortfalls, trends. Only instalments due and paid before the application date are used. |
| **Depends on** | Step 1. |
| **Output** | `f_inst` (one row per client). |

In [ ]:
# STEP 2a - installment behaviour per client (only instalments due and paid BEFORE the application: days < 0)
def trends(df, prefix, metrics):
    for m in metrics:                     # recent minus older window
        for r, o, n in [("l6m", "older6m", "trend6m"), ("l12m", "older12m", "trend12m")]:
            df[f"{prefix}{m}_{n}"] = df[f"{prefix}{m}_{r}"] - df[f"{prefix}{m}_{o}"]
    return df

con.execute('''CREATE OR REPLACE TEMP TABLE inst AS
  SELECT SK_ID_CURR, SK_ID_PREV, MAX(AMT_INSTALMENT) amt_due, SUM(COALESCE(AMT_PAYMENT,0)) amt_paid,
         MIN(DAYS_INSTALMENT) due, MAX(DAYS_ENTRY_PAYMENT) last_pay
  FROM installments_payments WHERE DAYS_INSTALMENT < 0 AND (DAYS_ENTRY_PAYMENT IS NULL OR DAYS_ENTRY_PAYMENT < 0)
  GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER''')
base = '''(SELECT *, last_pay - due AS days_diff,
             CASE WHEN last_pay IS NULL THEN NULL WHEN last_pay > due THEN 1.0 ELSE 0.0 END AS late,
             CASE WHEN last_pay IS NULL THEN NULL ELSE GREATEST(last_pay-due,0) END AS dpd,
             CASE WHEN amt_due > 0 THEN (CASE WHEN amt_paid < amt_due - 1.0 THEN 1.0 ELSE 0.0 END) END AS under FROM inst)'''
def w(cond, s):
    return f'''AVG(CASE WHEN {cond} THEN late END) inst_late_rate_{s}, AVG(CASE WHEN {cond} THEN dpd END) inst_mean_days_late_{s},
               MAX(CASE WHEN {cond} THEN dpd END) inst_max_days_late_{s}, AVG(CASE WHEN {cond} THEN under END) inst_underpaid_share_{s},
               SUM(CASE WHEN {cond} THEN amt_paid END)/NULLIF(SUM(CASE WHEN {cond} THEN amt_due END),0) inst_paid_ratio_{s},
               COUNT(CASE WHEN {cond} THEN 1 END) inst_n_due_{s}'''
f_inst = con.execute(f'''SELECT SK_ID_CURR, COUNT(DISTINCT SK_ID_PREV) inst_n_contracts, AVG(days_diff) inst_mean_days_diff_signed,
   {w('TRUE','all')}, {w('due>=-180','l6m')}, {w('due<-180','older6m')}, {w('due>=-365','l12m')}, {w('due<-365','older12m')}
   FROM {base} GROUP BY 1''').df()
f_inst = trends(f_inst, "inst_", ["late_rate", "mean_days_late", "underpaid_share", "paid_ratio"])
print("installment features:", f_inst.shape)

### Step 2b. Credit-card features

| | |
|---|---|
| **Input** | `credit_card_balance`. |
| **Purpose** | Per-client card utilisation, drawings and payment behaviour in the months before the application. |
| **Depends on** | Step 1. |
| **Output** | `f_cc`. |

In [ ]:
# STEP 2 - credit-card behaviour per client (months before the application only)
base = '''(SELECT *, AMT_BALANCE/NULLIF(AMT_CREDIT_LIMIT_ACTUAL,0) util, COALESCE(AMT_PAYMENT_TOTAL_CURRENT,AMT_PAYMENT_CURRENT,0) pay,
          COALESCE(AMT_DRAWINGS_ATM_CURRENT,0) atm, COALESCE(AMT_DRAWINGS_CURRENT,0) drawn, CASE WHEN AMT_INST_MIN_REGULARITY>0 THEN 1 END has_min
          FROM credit_card_balance WHERE MONTHS_BALANCE < 0)'''
def w(cond, s):
    return f'''AVG(CASE WHEN {cond} THEN util END) cc_util_mean_{s}, MAX(CASE WHEN {cond} THEN util END) cc_util_max_{s},
       AVG(CASE WHEN {cond} AND has_min=1 THEN (CASE WHEN pay>0 AND pay<=1.05*AMT_INST_MIN_REGULARITY THEN 1.0 ELSE 0.0 END) END) cc_minpay_only_share_{s},
       AVG(CASE WHEN {cond} AND has_min=1 THEN (CASE WHEN pay<AMT_INST_MIN_REGULARITY THEN 1.0 ELSE 0.0 END) END) cc_below_min_share_{s},
       SUM(CASE WHEN {cond} THEN atm END)/NULLIF(SUM(CASE WHEN {cond} THEN drawn END),0) cc_atm_share_{s},
       MAX(CASE WHEN {cond} THEN SK_DPD END) cc_max_dpd_{s}, AVG(CASE WHEN {cond} THEN SK_DPD END) cc_mean_dpd_{s},
       AVG(CASE WHEN {cond} THEN (CASE WHEN SK_DPD>0 THEN 1.0 ELSE 0.0 END) END) cc_share_months_dpd_{s}, COUNT(CASE WHEN {cond} THEN 1 END) cc_n_months_{s}'''
f_cc = con.execute(f'''SELECT SK_ID_CURR, COUNT(DISTINCT SK_ID_PREV) cc_n_cards, MAX(SK_DPD_DEF) cc_max_dpd_def,
  AVG(CASE WHEN SK_DPD_DEF>0 THEN 1.0 ELSE 0.0 END) cc_share_months_dpd_def,
  {w('TRUE','all')}, {w('MONTHS_BALANCE>=-6','l6m')}, {w('MONTHS_BALANCE<-6','older6m')}, {w('MONTHS_BALANCE>=-12','l12m')}, {w('MONTHS_BALANCE<-12','older12m')}
  FROM {base} GROUP BY 1''').df()
latest = con.execute('''WITH m AS (SELECT SK_ID_CURR, SK_ID_PREV, MAX(MONTHS_BALANCE) mb FROM credit_card_balance WHERE MONTHS_BALANCE<0 GROUP BY 1,2)
  SELECT c.SK_ID_CURR,
    SUM(CASE WHEN c.MONTHS_BALANCE>=-3 AND c.NAME_CONTRACT_STATUS='Active' THEN COALESCE(c.AMT_INST_MIN_REGULARITY,0) ELSE 0 END) cc_current_min_payment,
    SUM(CASE WHEN c.MONTHS_BALANCE>=-3 AND c.NAME_CONTRACT_STATUS='Active' THEN c.AMT_BALANCE ELSE 0 END) cc_current_balance,
    SUM(CASE WHEN c.MONTHS_BALANCE>=-3 AND c.NAME_CONTRACT_STATUS='Active' THEN c.AMT_CREDIT_LIMIT_ACTUAL ELSE 0 END) cc_current_limit,
    SUM(CASE WHEN c.MONTHS_BALANCE>=-3 AND c.NAME_CONTRACT_STATUS='Active' THEN 1 ELSE 0 END) cc_n_active_recent
  FROM credit_card_balance c JOIN m ON c.SK_ID_PREV=m.SK_ID_PREV AND c.MONTHS_BALANCE=m.mb GROUP BY 1''').df()
f_cc = f_cc.merge(latest, how="left", on="SK_ID_CURR")
f_cc["cc_current_util"] = f_cc["cc_current_balance"] / f_cc["cc_current_limit"].replace(0, np.nan)
f_cc = trends(f_cc, "cc_", ["util_mean", "minpay_only_share", "atm_share", "mean_dpd"])
print("credit-card features:", f_cc.shape)

### Step 2c. Credit-bureau features

| | |
|---|---|
| **Input** | `bureau`, `bureau_balance`. |
| **Purpose** | Per-client external credit history: active debt, overdue amounts, history status. Records updated after the application date are excluded. |
| **Depends on** | Step 1. |
| **Output** | `f_bur`, `f_bbx`. |

In [ ]:
# STEP 2c - credit bureau per client; rows updated AFTER the application (DAYS_CREDIT_UPDATE > 0) are dropped
con.execute('''CREATE OR REPLACE TEMP TABLE b AS SELECT *, CASE WHEN CREDIT_ACTIVE='Active' THEN 1 ELSE 0 END act,
     GREATEST(COALESCE(AMT_CREDIT_SUM_DEBT,0),0) debt,
     CASE WHEN DAYS_CREDIT_ENDDATE BETWEEN 30 AND 10957 THEN DAYS_CREDIT_ENDDATE/30.4375 END months_left
   FROM bureau WHERE DAYS_CREDIT_UPDATE <= 0''')
con.execute('''CREATE OR REPLACE TEMP TABLE b2 AS SELECT *,
   CASE WHEN act=1 AND AMT_ANNUITY > 0 THEN AMT_ANNUITY END ann_raw,
   CASE WHEN act=1 AND AMT_ANNUITY > 0 THEN AMT_ANNUITY WHEN act=1 AND debt > 0 AND months_left IS NOT NULL THEN debt/months_left END ann_est,
   CASE WHEN act=1 AND AMT_ANNUITY > 0 THEN AMT_ANNUITY WHEN act=1 AND debt > 0 AND months_left IS NOT NULL THEN debt/months_left
        WHEN act=1 AND debt > 0 THEN debt*0.05 END ann_est_fallback FROM b''')
types = ["Consumer credit", "Credit card", "Car loan", "Mortgage", "Microloan"]
tsql = ", ".join(f"SUM(CASE WHEN CREDIT_TYPE='{t}' THEN 1 ELSE 0 END) \"bur_n_type_{t.replace(' ','_').lower()}\"" for t in types)
f_bur = con.execute(f'''SELECT SK_ID_CURR, COUNT(*) bur_n_credits, SUM(act) bur_n_active,
  SUM(CASE WHEN CREDIT_ACTIVE='Closed' THEN 1 ELSE 0 END) bur_n_closed, SUM(CASE WHEN CREDIT_ACTIVE IN ('Sold','Bad debt') THEN 1 ELSE 0 END) bur_n_sold_or_bad_debt,
  AVG(act*1.0) bur_active_share, SUM(CASE WHEN act=1 THEN debt END) bur_active_debt_sum, SUM(CASE WHEN act=1 THEN AMT_CREDIT_SUM END) bur_active_credit_sum,
  SUM(CASE WHEN act=1 THEN debt END)/NULLIF(SUM(CASE WHEN act=1 THEN AMT_CREDIT_SUM END),0) bur_active_debt_to_credit,
  SUM(debt)/NULLIF(SUM(AMT_CREDIT_SUM),0) bur_all_debt_to_credit, SUM(CASE WHEN act=1 THEN AMT_CREDIT_SUM_LIMIT END) bur_active_limit_sum,
  SUM(AMT_CREDIT_SUM_OVERDUE) bur_overdue_sum, MAX(CREDIT_DAY_OVERDUE) bur_max_day_overdue,
  SUM(CASE WHEN COALESCE(AMT_CREDIT_SUM_OVERDUE,0)>0 THEN 1 ELSE 0 END) bur_n_with_overdue, MAX(AMT_CREDIT_MAX_OVERDUE) bur_max_overdue_amt,
  SUM(CNT_CREDIT_PROLONG) bur_n_prolong, -MAX(DAYS_CREDIT) bur_days_since_last_credit, -MIN(DAYS_CREDIT) bur_days_since_first_credit,
  -MAX(CASE WHEN act=1 THEN DAYS_CREDIT END) bur_days_since_last_active_credit, SUM(CASE WHEN DAYS_CREDIT>=-365 THEN 1 ELSE 0 END) bur_n_credits_l12m,
  SUM(CASE WHEN DAYS_CREDIT>=-730 THEN 1 ELSE 0 END) bur_n_credits_l24m, -MAX(DAYS_CREDIT_UPDATE) bur_days_since_last_update,
  COUNT(DISTINCT CREDIT_TYPE) bur_n_credit_types, {tsql},
  SUM(ann_raw) bur_annuity_raw_active_sum, SUM(CASE WHEN ann_raw IS NOT NULL THEN 1 ELSE 0 END) bur_annuity_raw_n,
  SUM(ann_est) bur_annuity_est_active_sum, SUM(CASE WHEN ann_est IS NOT NULL THEN 1 ELSE 0 END) bur_annuity_est_n,
  SUM(ann_est_fallback) bur_annuity_est_fallback_active_sum FROM b2 GROUP BY 1''').df()
f_bur["bur_annuity_est_coverage_of_active"] = f_bur["bur_annuity_est_n"] / f_bur["bur_n_active"].replace(0, np.nan)
f_bur["bur_annuity_raw_coverage_of_active"] = f_bur["bur_annuity_raw_n"] / f_bur["bur_n_active"].replace(0, np.nan)
def w(cond, s):
    k = "STATUS IN ('0','1','2','3','4','5')"
    return f'''SUM(CASE WHEN {cond} AND STATUS IN ('1','2','3','4','5') THEN 1.0 ELSE 0 END)/NULLIF(SUM(CASE WHEN {cond} AND {k} THEN 1.0 ELSE 0 END),0) bb_status1to5_share_{s},
      SUM(CASE WHEN {cond} AND STATUS IN ('3','4','5') THEN 1.0 ELSE 0 END)/NULLIF(SUM(CASE WHEN {cond} AND {k} THEN 1.0 ELSE 0 END),0) bb_status3to5_share_{s},
      MAX(CASE WHEN {cond} AND {k} THEN CAST(STATUS AS INT) END) bb_max_status_{s}, SUM(CASE WHEN {cond} THEN 1 ELSE 0 END) bb_n_months_{s}'''
f_bb = con.execute(f'''SELECT b.SK_ID_CURR, {w('TRUE','all')}, {w('MONTHS_BALANCE>=-12','l12m')}, {w('MONTHS_BALANCE<-12','older12m')},
     {w('MONTHS_BALANCE>=-6','l6m')}, {w('MONTHS_BALANCE<-6','older6m')},
     SUM(CASE WHEN STATUS='X' THEN 1.0 ELSE 0 END)/COUNT(*) bb_unknown_share, COUNT(DISTINCT bb.SK_ID_BUREAU) bb_n_credits
   FROM bureau_balance bb JOIN b ON bb.SK_ID_BUREAU=b.SK_ID_BUREAU WHERE bb.MONTHS_BALANCE<=0 GROUP BY 1''').df()
f_bb = trends(f_bb, "bb_", ["status1to5_share"])
f_bur = f_bur.merge(f_bb, how="left", on="SK_ID_CURR")
# extra bureau_balance features from the feasibility study (bbx_): recency of the last late month etc.
f_bbx = con.execute('''WITH bb AS (SELECT b.SK_ID_CURR, bb.SK_ID_BUREAU, bb.MONTHS_BALANCE m, bb.STATUS s
                   FROM bureau_balance bb JOIN bureau b USING (SK_ID_BUREAU) WHERE bb.MONTHS_BALANCE <= 0),
  per AS (SELECT SK_ID_CURR, SK_ID_BUREAU, MAX(CASE WHEN s IN ('1','2','3','4','5') THEN 1 ELSE 0 END) ever_bad,
                 MAX(CASE WHEN s IN ('2','3','4','5') THEN 1 ELSE 0 END) ever_bad60 FROM bb GROUP BY 1,2)
  SELECT a.SK_ID_CURR, -MAX(CASE WHEN s IN ('1','2','3','4','5') THEN m END) bbx_months_since_last_bad,
    -MAX(CASE WHEN s IN ('2','3','4','5') THEN m END) bbx_months_since_last_bad60,
    AVG(CASE WHEN s IN ('2','3','4','5') THEN 1.0 WHEN s IN ('0','1') THEN 0.0 END) bbx_status2to5_share_known,
    AVG(CASE WHEN s = 'C' THEN 1.0 ELSE 0.0 END) bbx_closed_month_share,
    MAX(p.n_ever_bad) bbx_n_credits_ever_bad, MAX(p.share_ever_bad) bbx_share_credits_ever_bad, MAX(p.n_ever_bad60) bbx_n_credits_ever_bad60
  FROM bb a JOIN (SELECT SK_ID_CURR, SUM(ever_bad) n_ever_bad, AVG(ever_bad) share_ever_bad, SUM(ever_bad60) n_ever_bad60 FROM per GROUP BY 1) p USING (SK_ID_CURR)
  GROUP BY a.SK_ID_CURR''').df()
print("bureau features:", f_bur.shape, "| extra bureau_balance:", f_bbx.shape)

### Step 2d. Previous-application features

| | |
|---|---|
| **Input** | `previous_application`. |
| **Purpose** | Per-client history of earlier applications with Home Credit: approvals, refusals, typical amounts and terms. |
| **Depends on** | Step 1. |
| **Output** | `f_prev`. |

In [ ]:
# STEP 2d - previous applications per client (all decided before the current application)
S = 365243   # 'not applicable' code used in DAYS_* columns
statuses = ["Approved", "Refused", "Canceled", "Unused offer"]
s_sql = ", ".join(f"SUM(CASE WHEN NAME_CONTRACT_STATUS='{s}' THEN 1 ELSE 0 END) \"prev_n_{s.split()[0].lower()}\"" for s in statuses)
reasons = ["LIMIT", "SCOFR", "SCO", "XAP", "VERIF", "XNA", "HC", "SYSTEM"]
r_sql = ", ".join(f"SUM(CASE WHEN NAME_CONTRACT_STATUS='Refused' AND CODE_REJECT_REASON='{r}' THEN 1 ELSE 0 END) \"prev_refused_{r}\"" for r in reasons)
tp = ", ".join(f"SUM(CASE WHEN NAME_CONTRACT_TYPE='{t}' THEN 1 ELSE 0 END) \"prev_n_{t.split()[0].lower()}\"" for t in ["Consumer loans", "Cash loans", "Revolving loans"])
act = f"NAME_CONTRACT_STATUS='Approved' AND DAYS_FIRST_DUE<>{S} AND DAYS_FIRST_DUE IS NOT NULL AND DAYS_LAST_DUE={S} AND DAYS_TERMINATION={S}"
f_prev = con.execute(f'''SELECT SK_ID_CURR, COUNT(*) prev_n_total, {s_sql}, {r_sql}, {tp},
  -MAX(DAYS_DECISION) prev_days_since_last_decision, -MAX(CASE WHEN NAME_CONTRACT_STATUS='Approved' THEN DAYS_DECISION END) prev_days_since_last_approved,
  -MAX(CASE WHEN NAME_CONTRACT_STATUS='Refused' THEN DAYS_DECISION END) prev_days_since_last_refused, -MIN(DAYS_DECISION) prev_days_since_first_decision,
  SUM(CASE WHEN DAYS_DECISION>=-365 THEN 1 ELSE 0 END) prev_n_apps_l12m, SUM(CASE WHEN DAYS_DECISION>=-180 THEN 1 ELSE 0 END) prev_n_apps_l6m,
  SUM(CASE WHEN DAYS_DECISION>=-365 AND NAME_CONTRACT_STATUS='Refused' THEN 1 ELSE 0 END) prev_n_refused_l12m,
  AVG(CASE WHEN NAME_CONTRACT_STATUS='Approved' AND AMT_CREDIT>0 THEN AMT_APPLICATION/AMT_CREDIT END) prev_req_over_granted_mean,
  MIN(CASE WHEN NAME_CONTRACT_STATUS='Approved' AND AMT_CREDIT>0 THEN AMT_APPLICATION/AMT_CREDIT END) prev_req_over_granted_min,
  MAX(CASE WHEN NAME_CONTRACT_STATUS='Approved' AND AMT_CREDIT>0 THEN AMT_APPLICATION/AMT_CREDIT END) prev_req_over_granted_max,
  AVG(CASE WHEN NAME_CONTRACT_STATUS='Approved' AND AMT_CREDIT>0 AND AMT_APPLICATION>0 THEN (CASE WHEN AMT_CREDIT<AMT_APPLICATION THEN 1.0 ELSE 0 END) END) prev_share_granted_below_requested,
  AVG(CASE WHEN NAME_CONTRACT_STATUS='Approved' THEN CNT_PAYMENT END) prev_mean_term_months,
  AVG(CASE WHEN NAME_CONTRACT_STATUS='Approved' THEN AMT_ANNUITY END) prev_mean_annuity_approved,
  AVG(CASE WHEN NAME_CONTRACT_STATUS='Approved' THEN AMT_CREDIT END) prev_mean_credit_approved,
  AVG(CASE WHEN NAME_YIELD_GROUP='high' THEN 1.0 WHEN NAME_YIELD_GROUP IN ('low_normal','low_action','middle') THEN 0.0 END) prev_share_high_yield,
  SUM(CASE WHEN {act} THEN AMT_ANNUITY END) hc_active_term_annuity, SUM(CASE WHEN {act} THEN 1 ELSE 0 END) hc_n_active_term_loans,
  SUM(CASE WHEN {act} AND DAYS_LAST_DUE_1ST_VERSION>0 AND DAYS_LAST_DUE_1ST_VERSION<>{S} THEN AMT_ANNUITY*DAYS_LAST_DUE_1ST_VERSION/30.4375 END) hc_active_remaining_scheduled
  FROM previous_application GROUP BY 1''').df()
f_prev["prev_refusal_rate"] = f_prev["prev_n_refused"] / f_prev["prev_n_total"]
f_prev["prev_approval_rate"] = f_prev["prev_n_approved"] / f_prev["prev_n_total"]
f_prev["prev_refusal_rate_decided_only"] = f_prev["prev_n_refused"] / (f_prev["prev_n_approved"] + f_prev["prev_n_refused"]).replace(0, np.nan)
print("previous-application features:", f_prev.shape)

### Step 2e. Affordability features

| | |
|---|---|
| **Input** | Application fields and the aggregates of steps 2a to 2d. |
| **Purpose** | Instalment-to-income, debt-to-income and residual-income ratios, with and without known existing obligations. |
| **Depends on** | Steps 2a to 2d. |
| **Output** | `f_aff`. |

In [ ]:
# STEP 2e - affordability ratios (instalment / income etc.). Missing record = no known obligation of that kind (0).
app_all = con.execute('''SELECT SK_ID_CURR, AMT_INCOME_TOTAL, AMT_ANNUITY, AMT_CREDIT, AMT_GOODS_PRICE, CNT_FAM_MEMBERS, OCCUPATION_TYPE, NAME_EDUCATION_TYPE
                         FROM application_train UNION ALL
                         SELECT SK_ID_CURR, AMT_INCOME_TOTAL, AMT_ANNUITY, AMT_CREDIT, AMT_GOODS_PRICE, CNT_FAM_MEMBERS, OCCUPATION_TYPE, NAME_EDUCATION_TYPE
                         FROM application_test''').df()
a = (app_all.merge(f_bur[["SK_ID_CURR", "bur_annuity_raw_active_sum", "bur_annuity_est_active_sum", "bur_annuity_est_fallback_active_sum", "bur_active_debt_sum"]], how="left", on="SK_ID_CURR")
            .merge(f_prev[["SK_ID_CURR", "hc_active_term_annuity", "hc_active_remaining_scheduled"]], how="left", on="SK_ID_CURR")
            .merge(f_cc[["SK_ID_CURR", "cc_current_min_payment", "cc_current_balance"]], how="left", on="SK_ID_CURR"))
for c in ["bur_annuity_raw_active_sum", "bur_annuity_est_active_sum", "bur_annuity_est_fallback_active_sum", "bur_active_debt_sum",
          "hc_active_term_annuity", "hc_active_remaining_scheduled", "cc_current_min_payment", "cc_current_balance"]:
    a[c] = a[c].fillna(0).astype(float)
inc, ann = a["AMT_INCOME_TOTAL"].astype(float), a["AMT_ANNUITY"].astype(float)
hc = a["hc_active_term_annuity"] + a["cc_current_min_payment"]
f_aff = pd.DataFrame({"SK_ID_CURR": a["SK_ID_CURR"]})
f_aff["aff_annuity_to_income"] = ann / inc
f_aff["aff_credit_to_income"] = a["AMT_CREDIT"] / inc
f_aff["aff_credit_to_annuity_term"] = a["AMT_CREDIT"] / ann
f_aff["aff_credit_to_goods"] = a["AMT_CREDIT"] / a["AMT_GOODS_PRICE"]
f_aff["aff_dti_plus_bureau_raw"] = (ann + a["bur_annuity_raw_active_sum"]) / inc
f_aff["aff_dti_plus_bureau_est"] = (ann + a["bur_annuity_est_active_sum"]) / inc
f_aff["aff_dti_plus_bureau_est_fallback"] = (ann + a["bur_annuity_est_fallback_active_sum"]) / inc
f_aff["aff_dti_plus_hc_active"] = (ann + hc) / inc
f_aff["aff_dti_total_raw"] = (ann + a["bur_annuity_raw_active_sum"] + hc) / inc
f_aff["aff_dti_total_est"] = (ann + a["bur_annuity_est_active_sum"] + hc) / inc
f_aff["aff_dti_total_est_fallback"] = (ann + a["bur_annuity_est_fallback_active_sum"] + hc) / inc
f_aff["aff_existing_oblig_est_to_income"] = (a["bur_annuity_est_active_sum"] + hc) / inc
f_aff["aff_bureau_debt_to_income"] = a["bur_active_debt_sum"] / inc
f_aff["aff_total_debt_to_income"] = (a["bur_active_debt_sum"] + a["cc_current_balance"] + a["hc_active_remaining_scheduled"]) / inc
f_aff["aff_total_debt_incl_new_to_income"] = (a["bur_active_debt_sum"] + a["cc_current_balance"] + a["hc_active_remaining_scheduled"] + a["AMT_CREDIT"]) / inc
fam = a["CNT_FAM_MEMBERS"].astype(float).replace(0, np.nan)
f_aff["aff_residual_per_member_new_only"] = (inc - ann) / fam
f_aff["aff_residual_per_member_total_raw"] = (inc - ann - a["bur_annuity_raw_active_sum"] - hc) / fam
f_aff["aff_residual_per_member_total_est"] = (inc - ann - a["bur_annuity_est_active_sum"] - hc) / fam
f_aff["aff_income_per_member"] = inc / fam
lg = np.log(inc.clip(inc.quantile(0.005), inc.quantile(0.995)))
f_aff["aff_income_log_winsor"] = lg
grp = a["OCCUPATION_TYPE"].fillna("__missing__") + "|" + a["NAME_EDUCATION_TYPE"]
size = grp.map(grp.value_counts())
f_aff["aff_income_peer_group_size"] = size.astype(float)
f_aff["aff_income_pctile_occ_edu"] = lg.groupby(grp).rank(pct=True).where(size >= 30, lg.groupby(a["NAME_EDUCATION_TYPE"]).rank(pct=True))
print("affordability features:", f_aff.shape)

### Step 2f. Assemble the feature matrix

| | |
|---|---|
| **Input** | `application_train`, `application_test` and all feature tables above. |
| **Purpose** | One row per application and one column per feature, with consistent categorical levels in train and test. Gender and age are excluded from the model inputs according to the settings. |
| **Depends on** | Steps 2a to 2e. |
| **Output** | `TR`, `TE`, `y`, `FEATS`, `FOLDS`. |

In [ ]:
# STEP 2f - one row per application: raw application fields + all aggregates; same for application_test
def assemble(table):
    df = con.execute(f"SELECT * FROM {table} ORDER BY SK_ID_CURR").df()
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].astype(float).where(df["DAYS_EMPLOYED"] != 365243)   # 'not applicable' -> missing
    for f in [f_bur, f_prev, f_inst, f_cc, f_aff, f_bbx]:
        df = df.merge(f, how="left", on="SK_ID_CURR")
    return df
TR, TE = assemble("application_train"), assemble("application_test")
if SMOKE_ROWS:
    TR = TR.sample(SMOKE_ROWS, random_state=SEED).sort_values("SK_ID_CURR").reset_index(drop=True)
y = TR.pop("TARGET").astype(int).values
CAT = [c for c in TR.columns if not pd.api.types.is_numeric_dtype(TR[c])]
for c in CAT:                                     # same category levels in train and test
    levels = sorted(set(TR[c].dropna().astype(str)) | set(TE[c].dropna().astype(str)))
    TR[c] = pd.Categorical(TR[c].astype(object).where(TR[c].notna()), categories=levels)
    TE[c] = pd.Categorical(TE[c].astype(object).where(TE[c].notna()), categories=levels)
for c in TR.columns:
    if c not in CAT and c != "SK_ID_CURR":
        TR[c] = TR[c].astype("float32"); TE[c] = TE[c].astype("float32")
PROTECTED = (["CODE_GENDER"] if not USE_GENDER else []) + (["DAYS_BIRTH"] if not USE_AGE else [])
FEATS = [c for c in TR.columns if c not in ["SK_ID_CURR"] + PROTECTED]
EXT = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # same split as the analysis report
print(f"train {TR.shape}, test {TE.shape}, default rate {y.mean():.4f}, model features {len(FEATS)} (excluded: {PROTECTED or 'none'})")

### Data overview (EDA summary)

Compact checks that justify the evaluation design: class balance, size and missingness. The full exploratory analysis is delivered separately.

In [ ]:
# DATA OVERVIEW - class balance, size, missingness and the types of model inputs
miss = TR[FEATS].isna().mean()
print(f"applications: {len(TR):,} | model features: {len(FEATS)} ({len(CAT)} categorical)")
print(f"default rate: {y.mean():.2%} -> imbalanced target: accuracy is misleading, so AUC / PR-AUC / calibration are used")
print(f"features with more than 50% missing: {(miss > 0.5).sum()} | with more than 90% missing: {(miss > 0.9).sum()}")
print("share of defaulters in each fold:", [round(float(y[FOLDS[k][1]].mean()), 4) for k in range(5)], "(stratified, nearly identical)")
print("applications in test file (no labels):", f"{len(TE):,}")
miss.sort_values(ascending=False).head(8).round(3).to_frame("share missing")

### Step 3. Input plausibility and eligibility rules

| | |
|---|---|
| **Input** | Application fields (income, instalment, age, children, employment length). |
| **Purpose** | Rule-based checks for implausible inputs and an age-eligibility rule. Rules raise flags for the loan officer; they never reject an application automatically. |
| **Depends on** | Step 2f. |
| **Output** | `step3_input_rules.csv`. |

In [ ]:
# STEP 3 - plausibility rules + age eligibility rule (no model). Flags go to the loan officer; they never auto-reject.
age = -TR["DAYS_BIRTH"].astype(float) / 365.25
term_years = (TR["AMT_CREDIT"] / TR["AMT_ANNUITY"]).astype(float) / 12          # approximate term, ignores interest
yrs_emp = -TR["DAYS_EMPLOYED"].astype(float) / 365.25
RULES = {
    "annuity_exceeds_income": TR["AMT_ANNUITY"] > TR["AMT_INCOME_TOTAL"],
    "income_over_10M": TR["AMT_INCOME_TOTAL"] > 10_000_000,
    "children_over_8": TR["CNT_CHILDREN"] > 8,
    "credit_over_2x_goods": TR["AMT_CREDIT"] > 2 * TR["AMT_GOODS_PRICE"],
    "employed_longer_than_age_minus_14": yrs_emp > age - 14,
    f"age_below_{MIN_AGE}": age < MIN_AGE,
    f"age_at_maturity_over_{MAX_AGE_AT_MATURITY}": age + term_years > MAX_AGE_AT_MATURITY,
}
rule_tab = pd.DataFrame([dict(rule=k, n_flagged=int(v.sum()), share=v.mean(), default_rate_flagged=y[v.values].mean() if v.sum() else np.nan)
                         for k, v in RULES.items()])
rule_tab["overall_default_rate"] = y.mean()
rule_tab.to_csv(RUN / "step3_input_rules.csv", index=False)
rule_tab.round(4)

### Step 4. SAMA affordability cap

| | |
|---|---|
| **Input** | Instalment, income, income type, existing obligations. |
| **Purpose** | Computes the maximum instalment allowed by the salary-linked obligation cap (employees and retirees). Income is treated as monthly. |
| **Depends on** | Step 2f. |
| **Output** | `MAX_ANNUITY_SAMA`; share of applications above the cap. |

In [ ]:
# STEP 4 - SAMA-style affordability cap (policy rule). Income treated as monthly: if it were annual, ~99.8% of
# APPROVED loans would break the cap, which is implausible.
retired = (TR["NAME_INCOME_TYPE"].astype(str) == "Pensioner").values
CAP = np.where(retired, SAMA_CAP_RETIRED, SAMA_CAP_EMPLOYEE)
dti_new = (TR["AMT_ANNUITY"] / TR["AMT_INCOME_TOTAL"]).astype(float).values                       # new instalment only
dti_all = TR["aff_dti_total_est"].astype(float).values                                            # incl. existing obligations
over_new, over_all = dti_new > CAP, dti_all > CAP
print(f"new instalment above cap: {over_new.mean():.1%} (default {y[over_new].mean():.2%} vs {y[~over_new].mean():.2%})")
print(f"incl. existing obligations above cap: {over_all.mean():.1%} (default {y[over_all].mean():.2%} vs {y[~over_all].mean():.2%})")
# max instalment allowed by the cap -> used in step 11b to suggest an amount
MAX_ANNUITY_SAMA = CAP * TR["AMT_INCOME_TOTAL"].astype(float).values - np.maximum(dti_all - dti_new, 0) * TR["AMT_INCOME_TOTAL"].astype(float).values

## Experiment tools

Shared functions used by every model section: training and prediction for each algorithm, evaluation metrics, and checkpointing of every result to Drive.

| Algorithm | Role |
|---|---|
| Logistic regression | Linear reference model |
| LightGBM, XGBoost, CatBoost | Gradient-boosting models |
| Ensemble | Mean of the gradient-boosting models |

Early stopping uses 15% of the training part only; the held-out fold is never seen during training.

## Modelling and evaluation design

**Why these algorithms.**

| Algorithm | Role and reason |
|---|---|
| Logistic regression | Transparent linear **baseline**. It shows how much non-linear structure adds. |
| LightGBM | Gradient boosting with fast training, native categorical handling and exact SHAP contributions; the usual strong model for tabular credit data. Also supports monotone constraints. |
| XGBoost | A second boosting implementation with different tree growth and regularisation; GPU-capable. |
| CatBoost | Boosting with ordered target statistics for categorical variables, which reduces overfitting on high-cardinality categories. |
| Ensemble | Mean of the three boosting models. Averaging different learners reduces variance; the cost is more complexity at serving time. |

**Validation.** 5-fold stratified cross-validation with a fixed seed, identical folds for every algorithm, so differences come from the model and not from the split. Every row is predicted by models that did not train on it (out-of-fold predictions). Early stopping uses a 15% slice of the training part only; the evaluation fold is never seen during training.

**Metrics and why.**

| Metric | Question it answers | Why it is needed here |
|---|---|---|
| AUC | Does the model rank defaulters above non-defaulters? | Threshold-free ranking quality; lanes depend on ranking |
| PR-AUC | How well are the rare defaulters captured? | More informative than AUC when the positive class is rare |
| Brier score | How close are the probabilities to the outcomes? | Penalises both ranking and calibration errors |
| ECE | Does a predicted 10% correspond to about 10% observed? | Probabilities feed the SAMA check, lanes and amount suggestions |

**Calibration.** Boosting scores are not guaranteed to be probabilities. Isotonic regression (non-parametric, monotone) is fitted cross-fitted: fitted on one half of the out-of-fold predictions and applied to the other half, so the reported calibration error is honest.

**Hyperparameters.** Moderate-capacity, fixed settings with early stopping are used for all boosting models so that the comparison is fair and reproducible. A tuning search is listed under next steps.

**Selection rule.** The algorithm with the best cross-validated AUC is a candidate; differences of about 0.002 AUC are within fold-to-fold noise, so a simpler model is preferred unless a more complex one is clearly better. The team's decision for Model 1 is recorded in step 6.

In [ ]:
# EXPERIMENT TOOLS - one function per algorithm; early stopping uses 15% of the TRAINING part only (test fold never seen)
def _prep_lr(X, cat):
    X = X.copy()
    num = [c for c in X.columns if c not in cat]
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), "NA")
    X[num] = X[num].replace([np.inf, -np.inf], np.nan).clip(-1e9, 1e9)
    pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
                             ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="NA"), OneHotEncoder(handle_unknown="ignore", min_frequency=100)), cat)])
    return X, pre

def fit(algo, X, yy, tr, monotone=None, n_iter=None, seed_k=0):
    # returns (model, best_iteration). If n_iter is given: no early stopping, train on ALL of tr (used for final models)
    cat = [c for c in CAT if c in X.columns]
    if n_iter is None:
        t, es = train_test_split(tr, test_size=0.15, stratify=yy[tr], random_state=SEED + seed_k)
    else:
        t, es = tr, None
    if algo == "logreg":
        Xl, pre = _prep_lr(X, cat)
        m = make_pipeline(pre, LogisticRegression(C=0.05, max_iter=500)).fit(Xl.iloc[tr], yy[tr])
        return m, None
    if algo == "lightgbm":
        p = dict(objective="binary", metric="auc", learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.5, reg_lambda=5.0, verbosity=-1, seed=SEED)
        if monotone is not None:
            p.update(monotone_constraints=monotone, monotone_constraints_method="advanced")
        if n_iter:
            return lgb.train(p, lgb.Dataset(X.iloc[t], yy[t]), n_iter), n_iter
        b = lgb.train(p, lgb.Dataset(X.iloc[t], yy[t]), 3000, valid_sets=[lgb.Dataset(X.iloc[es], yy[es])], callbacks=[lgb.early_stopping(100, verbose=False)])
        return b, b.best_iteration
    if algo == "xgboost":
        kw = dict(learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, tree_method="hist",
                  device="cuda" if USE_GPU else "cpu", enable_categorical=True, max_cat_to_onehot=1, eval_metric="auc", random_state=SEED,
                  monotone_constraints=tuple(monotone) if monotone is not None else None)
        Xx = X.replace([np.inf, -np.inf], np.nan)
        if n_iter:
            return xgb.XGBClassifier(n_estimators=n_iter, **kw).fit(Xx.iloc[t], yy[t]), n_iter
        m = xgb.XGBClassifier(n_estimators=3000, early_stopping_rounds=100, **kw).fit(Xx.iloc[t], yy[t], eval_set=[(Xx.iloc[es], yy[es])], verbose=False)
        return m, m.best_iteration + 1
    if algo == "catboost":
        Xc = _cb(X, cat)
        kw = dict(learning_rate=0.08, depth=6, l2_leaf_reg=5, random_seed=SEED, verbose=0,
                  task_type="GPU" if (USE_GPU and monotone is None) else "CPU", monotone_constraints=list(monotone) if monotone is not None else None)
        if n_iter:
            return CatBoostClassifier(iterations=n_iter, **kw).fit(Xc.iloc[t], yy[t], cat_features=cat), n_iter
        m = CatBoostClassifier(iterations=3000, eval_metric="AUC", od_type="Iter", od_wait=100, **kw)
        m.fit(Xc.iloc[t], yy[t], cat_features=cat, eval_set=(Xc.iloc[es], yy[es]), use_best_model=True)
        return m, m.get_best_iteration() + 1
    raise ValueError(algo)

def _cb(X, cat):
    Xc = X.copy()
    for c in cat:
        Xc[c] = Xc[c].astype(object).where(Xc[c].notna(), "NA").astype(str)
    return Xc.replace([np.inf, -np.inf], np.nan)

def predict(algo, m, X):
    cat = [c for c in CAT if c in X.columns]
    if algo == "logreg":
        return m.predict_proba(_prep_lr(X, cat)[0])[:, 1]
    if algo == "lightgbm":
        return m.predict(X)
    if algo == "xgboost":
        return m.predict_proba(X.replace([np.inf, -np.inf], np.nan))[:, 1]
    return m.predict_proba(_cb(X, cat))[:, 1]

def ece(p, yy):
    d = pd.qcut(p, 10, labels=False, duplicates="drop")
    t = pd.DataFrame({"d": d, "p": p, "y": yy}).groupby("d").agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(t.p - t.y) * t.n).sum() / t.n.sum())

def experiment(task, algo, X, yy, folds, monotone=None, keep_models=False):
    # train on each training part, predict the held-out part; saved to Drive so a disconnect never repeats finished work
    f_res, f_oof = RUN / "experiments" / f"{task}__{algo}.json", RUN / "experiments" / f"{task}__{algo}.npy"
    if f_res.exists():
        print(f"  {task:<14} {algo:<9} (loaded from Drive)"); return json.loads(f_res.read_text()), np.load(f_oof)
    oof, iters, t0 = np.full(len(yy), np.nan), [], time.time()
    for k in EVAL_FOLDS:
        tr, va = folds[k]
        m, it = fit(algo, X, yy, tr, monotone, seed_k=k)
        oof[va] = predict(algo, m, X.iloc[va]); iters.append(it)
        if keep_models and algo == "lightgbm":
            m.save_model(str(RUN / "models" / f"{task}__lightgbm__fold{k}.txt"))
    v = ~np.isnan(oof)
    r = dict(task=task, algo=algo, n_eval=int(v.sum()), auc=roc_auc_score(yy[v], oof[v]), pr_auc=average_precision_score(yy[v], oof[v]),
             brier=brier_score_loss(yy[v], oof[v]), ece=ece(oof[v], yy[v]), best_iter_mean=float(np.mean(iters)) if iters[0] else None,
             minutes=round((time.time() - t0) / 60, 1))
    np.save(f_oof, oof); f_res.write_text(json.dumps(r))
    print(f"  {task:<14} {algo:<9} AUC {r['auc']:.4f}  PR-AUC {r['pr_auc']:.4f}  {r['minutes']} min")
    return r, oof

def compare(task, X, yy, folds, algos, monotone=None, keep_models=False):
    rows, oofs = [], {}
    for a in algos:
        r, o = experiment(task, a, X, yy, folds, monotone, keep_models)
        rows.append(r); oofs[a] = o
    gb = [a for a in ("lightgbm", "xgboost", "catboost") if a in oofs]
    if len(gb) >= 2:
        e = np.mean([oofs[a] for a in gb], axis=0); v = ~np.isnan(e); oofs["ensemble"] = e
        rows.append(dict(task=task, algo="ensemble", n_eval=int(v.sum()), auc=roc_auc_score(yy[v], e[v]), pr_auc=average_precision_score(yy[v], e[v]),
                         brier=brier_score_loss(yy[v], e[v]), ece=ece(e[v], yy[v])))
    return pd.DataFrame(rows), oofs

def pick_winner(tab, prefer="lightgbm", margin=0.002):
    # LightGBM is preferred (SHAP, monotone, fast) unless another algorithm beats it by more than `margin` AUC
    t = tab.set_index("algo")
    best = t.auc.idxmax()
    return prefer if prefer in t.index and t.auc.max() - t.loc[prefer, "auc"] <= margin else best

In [ ]:
# FINAL-TRAINING HELPERS - used at the end of every model section. An 'ensemble' = mean of lightgbm + xgboost + catboost, each retrained on ALL rows.
import pickle
FINAL = {}                                  # task -> list of (algorithm, fitted model)
GB = ("lightgbm", "xgboost", "catboost")
def n_iter_of(task, algo):                  # number of trees = mean best iteration over the CV folds (no early-stopping set needed)
    p = RUN / "experiments" / f"{task}__{algo}.json"
    return int(round(json.loads(p.read_text())["best_iter_mean"])) if algo != "logreg" else None
def train_final(task, algo, X, yy, mono=None):
    members = [a for a in GB if a in ALGOS] if algo == "ensemble" else [algo]
    out = []
    for a in members:
        f = RUN / "final" / f"{task}__{a}.pkl"
        if f.exists() and not RETRAIN_FINAL:
            try:
                with open(f, "rb") as fh:
                    out.append((a, pickle.load(fh)))
                print(f"{task}: {a} loaded from Drive (already trained on 100%)"); continue
            except Exception as err:                 # saved with a different library version: retrain it
                print(f"{task}: {a} could not be loaded ({type(err).__name__}); retraining")
        t0 = time.time(); m, _ = fit(a, X, yy, np.arange(len(yy)), mono, n_iter=n_iter_of(task, a))
        with open(f, "wb") as fh:
            pickle.dump(m, fh)
        out.append((a, m)); print(f"{task}: {a} trained on {len(yy):,} rows in {(time.time()-t0)/60:.1f} min -> saved")
    FINAL[task] = out
    return out
def predict_final(members, X):
    return np.mean([predict(a, m, X) for a, m in members], axis=0)

# Section 2. Model 1: default risk

Estimates the probability that an applicant will not repay. Calibration, segment audit, confidence, triage lanes and explanations are all derived from this model.

### Step 5. Model 1 (main default-risk model): algorithm comparison

| | |
|---|---|
| **Input** | Full training set, all model features. |
| **Purpose** | Compares logistic regression, LightGBM, XGBoost, CatBoost and their ensemble with the same 5 folds. Every row is scored by a model that did not train on it. |
| **Depends on** | Step 2f. |
| **Output** | Out-of-fold predictions and metrics (AUC, PR-AUC, Brier, ECE) per algorithm. |

In [ ]:
# STEP 5 - MODEL 1 (main PD): compare algorithms on identical folds; LightGBM fold models are kept for steps 9 and 12
tab_main, oof_main = compare("1_main", TR[FEATS], y, FOLDS, ALGOS, keep_models=True)
tab_main.round(4)

### Step 6. Model 1: winner and probability calibration

| | |
|---|---|
| **Input** | Out-of-fold predictions of step 5. |
| **Purpose** | Selects the production algorithm and calibrates its probabilities with isotonic regression, so that a predicted 10% corresponds to an observed 10%. |
| **Depends on** | Step 5. |
| **Output** | `CALIBRATOR`; calibration error before and after. |

In [ ]:
# STEP 6 - choose the winner of model 1 and calibrate its probabilities (cross-fitted isotonic: fit on one half, apply to the other)
WIN_MAIN = "ensemble"                      # DECISION (user): serve the ensemble = mean of lightgbm + xgboost + catboost (AUC 0.7907)
SERVE_MAIN = WIN_MAIN                      # step 14 retrains all three members on 100% and averages them
p_raw = oof_main[SERVE_MAIN]; v = ~np.isnan(p_raw)                # calibrate exactly the model that will be served
idx = np.where(v)[0]; rng = np.random.default_rng(SEED); half = rng.random(len(idx)) < 0.5
p_cal = np.full(len(y), np.nan)
for fit_i, app_i in [(idx[half], idx[~half]), (idx[~half], idx[half])]:
    p_cal[app_i] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_raw[fit_i], y[fit_i]).predict(p_raw[app_i])
CALIBRATOR = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_raw[v], y[v])     # production calibrator (all OOF rows)
print(f"best: {WIN_MAIN} | served + calibrated: {SERVE_MAIN} | ECE raw {ece(p_raw[v], y[v]):.4f} -> calibrated {ece(p_cal[v], y[v]):.4f}")

### Step 7. Model 1: performance by segment

| | |
|---|---|
| **Input** | Out-of-fold predictions, gender and age band. |
| **Purpose** | Fairness audit. Gender and age are used only to measure performance by group, never as model inputs. |
| **Depends on** | Step 5. |
| **Output** | `step7_segment_auc.csv`. |

In [ ]:
# STEP 7 - segment AUC (gender / age band used ONLY to measure, never as model input when USE_GENDER/USE_AGE are False)
band = pd.cut(age, [0, 25, 35, 45, 55, 65, 200], right=False, labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"]).astype(str)
seg = []
for name, s in [("gender", TR["CODE_GENDER"].astype(str).values), ("age_band", band.values)]:
    for lv in sorted(pd.unique(s)):
        m = (s == lv) & v
        if m.sum() > 500:
            seg.append(dict(segment=name, group=lv, n=int(m.sum()), default_rate=y[m].mean(), auc=roc_auc_score(y[m], p_raw[m])))
pd.DataFrame(seg).to_csv(RUN / "step7_segment_auc.csv", index=False)
pd.DataFrame(seg).round(4)

### Step 8. Model 1: prediction confidence

| | |
|---|---|
| **Input** | The cross-validation LightGBM models of step 5; `application_test`. |
| **Purpose** | Measures the disagreement between the fold models for each applicant as an indicator of prediction stability. |
| **Depends on** | Step 5. |
| **Output** | Relative spread per applicant. |

In [ ]:
# STEP 8 - confidence = disagreement between the 5 LightGBM fold models (production view: every model scores every applicant)
fold_models = [lgb.Booster(model_file=str(RUN / "models" / f"1_main__lightgbm__fold{k}.txt")) for k in EVAL_FOLDS]
if len(fold_models) >= 2:
    P5 = np.vstack([m.predict(TE[FEATS]) for m in fold_models])      # scored on application_test (unseen clients)
    rel_sd = P5.std(axis=0, ddof=1) / P5.mean(axis=0)
    print(f"application_test: median relative spread {np.median(rel_sd):.3f}; 90th pct {np.percentile(rel_sd, 90):.3f}")
else:
    print("needs >= 2 evaluation folds (EVAL_FOLDS)")

### Step 9. Model 1: triage lanes

| | |
|---|---|
| **Input** | Calibrated probabilities of step 6. |
| **Purpose** | Defines the fast lane (lowest risk) and the strict-review lane (highest risk) by percentile cut-offs, and reports default rates and the age and gender mix of each lane. |
| **Depends on** | Step 6. |
| **Output** | `step9_lanes.csv`, `LANE_FAST_PD`, `LANE_HIGH_PD`. |

In [ ]:
# STEP 9 - triage lanes from calibrated PD; also who lands where (age / gender), for the audit page
pc = np.where(np.isnan(p_cal), p_raw, p_cal)
lanes = []
for q in (10, 20, 30, 40, 50):
    thr = np.nanpercentile(pc[v], q); lane = (pc <= thr) & v
    lanes.append(dict(lane="fast", pd_percentile=q, share_of_apps=lane.sum() / v.sum(), default_rate=y[lane].mean(), share_of_all_defaults=y[lane].sum() / y[v].sum()))
for q in (5, 10, 20):
    thr = np.nanpercentile(pc[v], 100 - q); lane = (pc > thr) & v
    lanes.append(dict(lane="high_risk", pd_percentile=100 - q, share_of_apps=lane.sum() / v.sum(), default_rate=y[lane].mean(), share_of_all_defaults=y[lane].sum() / y[v].sum()))
LANE_FAST_PD = float(np.nanpercentile(pc[v], 30)); LANE_HIGH_PD = float(np.nanpercentile(pc[v], 90))   # calibrated-PD cut-offs used by the website
lane_tab = pd.DataFrame(lanes); lane_tab.to_csv(RUN / "step9_lanes.csv", index=False)
fast = (pc <= np.nanpercentile(pc[v], 30)) & v
print("fast-lane share by age band:", {b: round(fast[(band.values == b) & v].mean(), 3) for b in ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]})
lane_tab.round(4)

### Step 10. Model 1: explanations

| | |
|---|---|
| **Input** | Model 1 (LightGBM, fold 0) and a validation sample. |
| **Purpose** | SHAP contributions mapped to fixed Arabic sentence templates. The text only restates the model's own contributions. |
| **Depends on** | Step 5. |
| **Output** | Top risk factors per applicant. |

In [ ]:
# STEP 10 - SHAP explanations with fixed sentence templates (the text only restates the model's own numbers)
k0 = EVAL_FOLDS[0]; m0 = lgb.Booster(model_file=str(RUN / "models" / f"1_main__lightgbm__fold{k0}.txt"))
va0 = np.random.default_rng(SEED).choice(FOLDS[k0][1], min(2000, len(FOLDS[k0][1])), replace=False)
phi = m0.predict(TR.iloc[va0][FEATS], pred_contrib=True)[:, :-1]
top5 = np.argsort(-np.abs(phi), axis=1)[:, :5]
freq = pd.Series(np.array(FEATS)[top5.ravel()]).value_counts().head(15)
AR = {"EXT_SOURCE_1": "الدرجة الخارجية 1", "EXT_SOURCE_2": "الدرجة الخارجية 2", "EXT_SOURCE_3": "الدرجة الخارجية 3", "AMT_ANNUITY": "القسط الشهري",
      "AMT_CREDIT": "مبلغ التمويل", "AMT_GOODS_PRICE": "سعر السلعة", "aff_credit_to_goods": "نسبة التمويل إلى سعر السلعة",
      "aff_credit_to_annuity_term": "مدة السداد التقريبية", "DAYS_EMPLOYED": "مدة العمل", "ORGANIZATION_TYPE": "جهة العمل", "OCCUPATION_TYPE": "المهنة",
      "NAME_EDUCATION_TYPE": "التعليم", "prev_mean_annuity_approved": "متوسط القسط في القروض السابقة", "prev_mean_term_months": "متوسط مدة القروض السابقة",
      "hc_active_remaining_scheduled": "المتبقي على القروض النشطة", "inst_late_rate_all": "نسبة الأقساط المتأخرة سابقاً", "OWN_CAR_AGE": "عمر السيارة",
      "bur_active_debt_to_credit": "نسبة الدين القائم في السجل الائتماني", "prev_refusal_rate": "نسبة الطلبات المرفوضة سابقاً"}
def explain(i):
    r = np.where(va0 == i)[0][0]
    return [f"{AR.get(FEATS[j], FEATS[j])}: {'رفع' if phi[r, j] > 0 else 'خفّض'} تقدير الخطر" for j in top5[r]]
print("most frequent reasons:", list(freq.index[:10]))
for line in explain(va0[0]):
    print(" -", line)

In [ ]:
# INTERPRETATION - Model 1 (all numbers computed from the results above)
t = tab_main.set_index("algo")
base, boosters = t.loc["logreg", "auc"], t.loc[[a for a in ("lightgbm", "xgboost", "catboost") if a in t.index], "auc"]
print(f"AUC: logistic regression {base:.4f} | boosting {boosters.min():.4f} to {boosters.max():.4f} | ensemble {t.loc['ensemble', 'auc']:.4f}")
print(f"-> boosting adds {boosters.max() - base:+.4f} AUC over the linear baseline; the spread between boosting algorithms is only {boosters.max() - boosters.min():.4f}")
print(f"-> ensemble vs best single model: {t.loc['ensemble', 'auc'] - boosters.max():+.4f} AUC, {t.loc['ensemble', 'pr_auc'] - t.loc[boosters.idxmax(), 'pr_auc']:+.4f} PR-AUC")
print(f"calibration (ECE): raw {ece(p_raw[v], y[v]):.4f} -> calibrated {ece(p_cal[v], y[v]):.4f}")
sg = pd.DataFrame(seg)
for name, g in sg.groupby("segment"):
    lo, hi = g.loc[g.auc.idxmin()], g.loc[g.auc.idxmax()]
    print(f"segment {name}: AUC ranges from {lo.auc:.4f} ({lo.group}) to {hi.auc:.4f} ({hi.group}); default rate {g.default_rate.min():.1%} to {g.default_rate.max():.1%}")
ln = lane_tab.set_index(["lane", "pd_percentile"])
print(f"fast lane (lowest 30% PD): {ln.loc[('fast', 30), 'share_of_apps']:.1%} of applications, default rate {ln.loc[('fast', 30), 'default_rate']:.2%} vs {y.mean():.2%} overall")
print(f"strict lane (highest 10% PD): default rate {ln.loc[('high_risk', 90), 'default_rate']:.1%}, holding {ln.loc[('high_risk', 90), 'share_of_all_defaults']:.1%} of all defaults")
fa = {b: fast[(band.values == b) & v].mean() for b in ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]}
print("fast-lane share by age band:", {k: round(float(x), 3) for k, x in fa.items()}, "-> a policy decision is needed if the gap is judged unfair")

### Step 11. Model 1: final training on 100% of the data

| | |
|---|---|
| **Input** | Full training set. |
| **Purpose** | Retrains the selected algorithm (all ensemble members) on every row with the tree counts found in cross-validation, then saves the models and the calibrator and scores `application_test`. |
| **Depends on** | Steps 5 and 6. |
| **Output** | `final/1_main__*.pkl`, `final/calibrator_1_main.pkl`, `final/application_test_scores.csv`. |

In [ ]:
# STEP 11 - FINAL MODEL 1: retrain the winner (ensemble) on ALL rows, save members + calibrator, score application_test
train_final("1_main", SERVE_MAIN, TR[FEATS], y)
with open(RUN / "final" / "calibrator_1_main.pkl", "wb") as fh:
    pickle.dump(CALIBRATOR, fh)
pd_test = CALIBRATOR.predict(predict_final(FINAL["1_main"], TE[FEATS]))
pd.DataFrame({"SK_ID_CURR": TE.SK_ID_CURR, "pd_calibrated": pd_test}).to_csv(RUN / "final" / "application_test_scores.csv", index=False)
print("application_test scored:", len(pd_test), "| mean PD", round(float(pd_test.mean()), 4), "| lanes: fast <=", round(LANE_FAST_PD, 4), "| high risk >=", round(LANE_HIGH_PD, 4))

# Section 3. Model 2: fallback without external scores

Estimates default risk when `EXT_SOURCE_1/2/3` are unavailable at decision time. Trained independently of Model 1.

### Step 12. Model 2 (fallback without external scores): algorithm comparison

| | |
|---|---|
| **Input** | Full training set without `EXT_SOURCE_1/2/3`. |
| **Purpose** | Risk model for applications where external scores are unavailable at decision time. Same folds and procedure as Model 1. |
| **Depends on** | Step 2f. Independent of Model 1. |
| **Output** | Out-of-fold predictions and metrics per algorithm. |

In [ ]:
# STEP 12 - MODEL 2 (fallback without external scores)
FEATS_FB = [c for c in FEATS if c not in EXT]
tab_fb, oof_fb = compare("2_fallback", TR[FEATS_FB], y, FOLDS, ALGOS)
WIN_FB = pick_winner(tab_fb)
print("winner:", WIN_FB); tab_fb.round(4)

In [ ]:
# INTERPRETATION - Model 2 (fallback)
a, b = tab_main.set_index("algo"), tab_fb.set_index("algo")
print(f"best AUC with external scores {a.auc.max():.4f} | without {b.auc.max():.4f} | cost of missing external scores {a.auc.max() - b.auc.max():.4f} AUC")
print("-> the fallback is usable but clearly weaker: external scores carry a large part of the signal" if a.auc.max() - b.auc.max() > 0.01 else "-> the fallback loses little: external scores add only a small amount of signal")

### Step 13. Model 2: final training on 100% of the data

| | |
|---|---|
| **Input** | Full training set without external scores. |
| **Purpose** | Retrains the selected algorithm on every row and fits its own isotonic calibrator. |
| **Depends on** | Step 12. |
| **Output** | `final/2_fallback__*.pkl`, `final/calibrator_2_fallback.pkl`. |

In [ ]:
# STEP 13 - FINAL MODEL 2 (fallback): retrain on ALL rows + its own isotonic calibrator (fit on the out-of-fold predictions)
train_final("2_fallback", WIN_FB, TR[FEATS_FB], y)
p_fb = oof_fb[WIN_FB]; vf = ~np.isnan(p_fb)
CALIBRATOR_FB = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_fb[vf], y[vf])
with open(RUN / "final" / "calibrator_2_fallback.pkl", "wb") as fh:
    pickle.dump(CALIBRATOR_FB, fh)
print("model 2 saved | ECE raw", round(ece(p_fb[vf], y[vf]), 4), "-> calibrated (in-sample)", round(ece(CALIBRATOR_FB.predict(p_fb[vf]), y[vf]), 4))

# Section 4. Model 3: monotone risk model and amount suggestion

Risk model with monotone constraints (a larger amount, instalment or debt ratio never lowers the predicted risk), used to suggest a smaller loan amount that satisfies the SAMA cap and the risk threshold.

### Step 14. Model 3 (monotone risk model): algorithm comparison

| | |
|---|---|
| **Input** | Full training set, with monotone constraints. |
| **Purpose** | Risk model in which a larger amount, instalment or debt ratio can never lower the predicted risk. Used for loan-amount suggestions. CatBoost is excluded because its monotone mode is CPU-only and impractically slow. |
| **Depends on** | Step 2f. Independent of Model 1. |
| **Output** | Out-of-fold predictions, metrics and the gap to Model 1. |

In [ ]:
# STEP 14 - MODEL 3 (monotone): risk can never go DOWN when amount / instalment / debt ratios go UP
UP = ["AMT_CREDIT", "AMT_ANNUITY", "aff_annuity_to_income", "aff_credit_to_income", "aff_dti_plus_bureau_raw", "aff_dti_plus_bureau_est",
      "aff_dti_plus_bureau_est_fallback", "aff_dti_plus_hc_active", "aff_dti_total_raw", "aff_dti_total_est", "aff_dti_total_est_fallback",
      "aff_total_debt_incl_new_to_income", "inst_late_rate_all", "inst_max_days_late_all", "bur_active_debt_to_credit", "cc_util_mean_all", "prev_refusal_rate"]
DOWN = ["aff_residual_per_member_new_only", "aff_residual_per_member_total_raw", "aff_residual_per_member_total_est"] + EXT
MONO = [1 if c in UP else (-1 if c in DOWN else 0) for c in FEATS]
tab_mono, oof_mono = compare("3_monotone", TR[FEATS], y, FOLDS, ALGOS_MONO, monotone=MONO, keep_models=True)
gap = tab_main.set_index("algo").loc[WIN_MAIN, "auc"] - tab_mono.set_index("algo").auc.max()
print(f"AUC gap main - monotone: {gap:+.4f} ->", "monotone can replace model 1" if gap <= 0.002 else "keep both")
tab_mono.round(4)

### Step 15. Model 3: loan-amount suggestion

| | |
|---|---|
| **Input** | Applicants above the risk threshold, monotone fold models, SAMA cap. |
| **Purpose** | For each applicant above the risk threshold, finds the largest reduced amount (down to -30%) that satisfies both the SAMA cap and the risk threshold. A suggestion is shown only if the risk reduction exceeds the model noise (2 standard deviations across fold models). |
| **Depends on** | Step 14 and Step 4. |
| **Output** | Share of applicants with a valid suggestion; `THR`. |

In [ ]:
# STEP 15 - suggested amount = largest credit (down to -30%) that satisfies BOTH the SAMA cap and the risk threshold
mono_models = {k: lgb.Booster(model_file=str(RUN / "models" / f"3_monotone__lightgbm__fold{k}.txt")) for k in EVAL_FOLDS}
pm = oof_mono["lightgbm"]; vm = ~np.isnan(pm)
THR = np.percentile(pm[vm], 80)                      # risk threshold = PD at an 80% approval rate (a policy choice)
fold_of = np.full(len(y), -1)
for k in EVAL_FOLDS:
    fold_of[FOLDS[k][1]] = k

def shrink(df, f):
    d = df.copy(); inc = d["AMT_INCOME_TOTAL"].astype(float).values; fam = d["CNT_FAM_MEMBERS"].astype(float).replace(0, np.nan).values
    c0, a0 = d["AMT_CREDIT"].astype(float).values, d["AMT_ANNUITY"].astype(float).values; c1, a1 = c0 * f, a0 * f
    d["AMT_CREDIT"], d["AMT_ANNUITY"] = c1, a1
    d["aff_annuity_to_income"], d["aff_credit_to_income"], d["aff_credit_to_goods"] = a1 / inc, c1 / inc, c1 / d["AMT_GOODS_PRICE"].astype(float).values
    for c in ["aff_dti_plus_bureau_raw", "aff_dti_plus_bureau_est", "aff_dti_plus_bureau_est_fallback", "aff_dti_plus_hc_active", "aff_dti_total_raw",
              "aff_dti_total_est", "aff_dti_total_est_fallback"]:
        d[c] = d[c].astype(float).values + (a1 - a0) / inc
    d["aff_total_debt_incl_new_to_income"] = d["aff_total_debt_incl_new_to_income"].astype(float).values + (c1 - c0) / inc
    for c in ["aff_residual_per_member_new_only", "aff_residual_per_member_total_raw", "aff_residual_per_member_total_est"]:
        d[c] = d[c].astype(float).values - (a1 - a0) / fam
    for c in d.columns:
        if c.startswith(("aff_", "AMT_")):
            d[c] = d[c].astype("float32")
    return d

elig = np.where(vm & (pm > THR) & TR["AMT_GOODS_PRICE"].notna().values & (TR["AMT_ANNUITY"].values > 0))[0]
S = np.sort(np.random.default_rng(SEED).choice(elig, min(3000, len(elig)), replace=False))
base = TR.iloc[S][FEATS]; ks = fold_of[S]
P0 = np.vstack([mono_models[k].predict(base) for k in EVAL_FOLDS]); own = [EVAL_FOLDS.index(k) for k in ks]
p0 = P0[own, np.arange(len(S))]
best_f = np.full(len(S), np.nan); gate = np.zeros(len(S), bool)
ann0 = TR["AMT_ANNUITY"].astype(float).values[S]; max_ann = MAX_ANNUITY_SAMA[S]
for f in np.round(np.arange(0.99, 0.695, -0.01), 2):
    P1 = np.vstack([mono_models[k].predict(shrink(base, f)) for k in EVAL_FOLDS]); p1 = P1[own, np.arange(len(S))]
    ok = np.isnan(best_f) & (p1 <= THR) & (ann0 * f <= max_ann)
    best_f[ok] = f
    if P1.shape[0] >= 2:
        gate[ok] = (p0 - p1)[ok] >= 2 * (P0 - P1).std(axis=0, ddof=1)[ok]
reached = ~np.isnan(best_f)
print(f"risk threshold (80% approval) = {THR:.3f}; above it: {len(elig) / vm.sum():.1%} of applicants")
print(f"a smaller amount within -30% meets SAMA cap AND threshold for {reached.mean():.1%} of them (median cut {np.nanmedian(1 - best_f):.0%}); "
      f"after the noise gate: {gate.mean():.1%}")

In [ ]:
# INTERPRETATION - Model 3 (monotone) and amount suggestion
t3 = tab_mono.set_index("algo")
print(f"monotone model AUC {t3.auc.max():.4f} vs unconstrained Model 1 {tab_main.auc.max():.4f}: the constraints change AUC by {t3.auc.max() - tab_main.auc.max():+.4f}")
print(f"applicants above the risk threshold: {len(elig) / vm.sum():.1%}; of a sample of them, a smaller amount (up to -30%) meets cap and threshold for {reached.mean():.1%}, and {gate.mean():.1%} remain after the noise gate")
print("-> the suggestion helps only a minority of high-risk applicants; for the rest the options are decline or a non-amount change" if gate.mean() < 0.5 else "-> the suggestion helps a large share of high-risk applicants")

### Step 16. Model 3: final training on 100% of the data

| | |
|---|---|
| **Input** | Full training set, with monotone constraints. |
| **Purpose** | Retrains LightGBM on every row. The five fold models remain saved for the noise gate of step 15. |
| **Depends on** | Steps 14 and 15. |
| **Output** | `final/3_monotone__lightgbm.pkl`. |

In [ ]:
# STEP 16 - FINAL MODEL 3 (monotone, LightGBM) on ALL rows; the 5 fold models stay in RUN/models for the noise gate
train_final("3_monotone", "lightgbm", TR[FEATS], y, MONO)
print("model 3 saved | risk threshold used for amount suggestions:", round(float(THR), 4))

# Section 5. Model 4: early warning

Estimates whether a running contract falls into arrears within the next 3 months, from monthly payment snapshots. Independent of Models 1 to 3.

### Step 17. Model 4 (early warning): algorithm comparison

| | |
|---|---|
| **Input** | `POS_CASH_balance`: one snapshot per contract-month, only for contracts not in arrears at that month. |
| **Purpose** | Predicts whether a running contract falls into arrears within the next 3 months. Folds are grouped by client so that no client appears in both training and test parts. Compared with a simple benchmark (maximum arrears in the last 3 months). |
| **Depends on** | Step 1. Independent of Models 1 to 3. |
| **Output** | Out-of-fold predictions, metrics and benchmark AUC. |

In [ ]:
# STEP 17 - MODEL 4 (early warning) from monthly POS data; folds grouped by client so one client is never in train and test
EW = con.execute('''
  WITH s AS (SELECT SK_ID_PREV, SK_ID_CURR, MONTHS_BALANCE m, SK_DPD dpd, SK_DPD_DEF dpd_def, CNT_INSTALMENT inst_total,
                    CNT_INSTALMENT_FUTURE inst_left, NAME_CONTRACT_STATUS st FROM POS_CASH_balance WHERE MONTHS_BALANCE < 0),
  w AS (SELECT *, MAX(dpd) OVER (PARTITION BY SK_ID_PREV ORDER BY m ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) dpd_max_l3,
     MAX(dpd) OVER (PARTITION BY SK_ID_PREV ORDER BY m ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) dpd_max_l6,
     MAX(dpd) OVER (PARTITION BY SK_ID_PREV ORDER BY m ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) dpd_max_all,
     AVG(CASE WHEN dpd>0 THEN 1.0 ELSE 0.0 END) OVER (PARTITION BY SK_ID_PREV ORDER BY m ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) dpd_share_l6,
     AVG(CASE WHEN dpd>0 THEN 1.0 ELSE 0.0 END) OVER (PARTITION BY SK_ID_PREV ORDER BY m ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) dpd_share_all,
     ROW_NUMBER() OVER (PARTITION BY SK_ID_PREV ORDER BY m) months_on_book,
     LAG(inst_left,3) OVER (PARTITION BY SK_ID_PREV ORDER BY m) - inst_left paid_down_l3,
     LEAD(m,3) OVER (PARTITION BY SK_ID_PREV ORDER BY m) m3,
     GREATEST(LEAD(dpd,1) OVER (PARTITION BY SK_ID_PREV ORDER BY m), LEAD(dpd,2) OVER (PARTITION BY SK_ID_PREV ORDER BY m),
              LEAD(dpd,3) OVER (PARTITION BY SK_ID_PREV ORDER BY m)) dpd_next3 FROM s),
  v AS (SELECT * FROM w WHERE m3 = m + 3 AND st <> 'Completed' AND dpd_next3 IS NOT NULL AND dpd = 0),
  r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY SK_ID_PREV ORDER BY hash(SK_ID_PREV, m, 42)) rn FROM v)
  SELECT SK_ID_CURR, dpd_def, dpd_max_l3, dpd_max_l6, dpd_max_all, dpd_share_l6, dpd_share_all, months_on_book, inst_total, inst_left,
         inst_left / NULLIF(inst_total,0) share_left, paid_down_l3, (dpd_next3 > 0)::INT y FROM r WHERE rn <= 2
  ORDER BY SK_ID_CURR, SK_ID_PREV, m''').df()
if SMOKE_ROWS:
    EW = EW[EW.SK_ID_CURR.isin(set(EW.SK_ID_CURR.drop_duplicates().sample(frac=0.05, random_state=SEED)))].reset_index(drop=True)
EW_FEATS = [c for c in EW.columns if c not in ("SK_ID_CURR", "y")]
for c in EW_FEATS:
    EW[c] = EW[c].astype("float32")
ye = EW.y.values
EW_FOLDS = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(EW, ye, EW.SK_ID_CURR.values))
tab_ew, oof_ew = compare("4_early_warning", EW[EW_FEATS], ye, EW_FOLDS, ALGOS)
ve = ~np.isnan(oof_ew[ALGOS[0]])
print(f"snapshots {len(EW):,}, base rate {ye.mean():.3f}; baseline (max DPD last 3 months) AUC {roc_auc_score(ye[ve], EW.dpd_max_l3.values[ve]):.4f}")
WIN_EW = pick_winner(tab_ew); tab_ew.round(4)

In [ ]:
# INTERPRETATION - Model 4 (early warning)
base_auc = roc_auc_score(ye[ve], EW.dpd_max_l3.values[ve])
t4 = tab_ew.set_index("algo")
print(f"benchmark (max days past due in the last 3 months) AUC {base_auc:.4f} | best model {t4.loc[[a for a in t4.index if a != 'ensemble'], 'auc'].max():.4f}")
print(f"base rate of new arrears within 3 months: {ye.mean():.2%}; PR-AUC of the best model {t4.pr_auc.max():.4f}")
print("-> improvement over the simple rule: " + ("clear" if t4.auc.max() - base_auc > 0.05 else "small") + "; absolute performance is modest, so it is a prioritisation aid, not a precise alarm")

### Step 18. Model 4: final training on 100% of the data

| | |
|---|---|
| **Input** | All snapshots. |
| **Purpose** | Retrains the selected algorithm on every snapshot. |
| **Depends on** | Step 17. |
| **Output** | `final/4_early_warning__*.pkl`. |

In [ ]:
# STEP 18 - FINAL MODEL 4 (early warning) on ALL snapshots
train_final("4_early_warning", WIN_EW, EW[EW_FEATS], ye)

### Step 19. Summary and deployment bundle

| | |
|---|---|
| **Input** | All saved experiments and final models. |
| **Purpose** | Collects every experiment into one table and writes the files the application needs: `manifest.json` (feature lists, categorical levels, thresholds, SAMA parameters, labels) and `serving.py` (scoring code). |
| **Depends on** | All previous steps. |
| **Output** | `summary.csv`, `final/manifest.json`, `final/serving.py`. |

In [ ]:
# STEP 19 - summary table + website bundle (manifest.json + serving.py next to the pickled models in RUN/final)
summary = pd.concat([tab_main, tab_fb, tab_mono, tab_ew], ignore_index=True)
summary.to_csv(RUN / "summary.csv", index=False)
MANIFEST = dict(
    seed=SEED, use_gender=USE_GENDER, use_age=USE_AGE,
    features={"main": FEATS, "fallback": FEATS_FB, "monotone": FEATS, "early_warning": EW_FEATS},
    categorical_levels={c: [str(x) for x in TR[c].cat.categories] for c in CAT if c in FEATS},
    final_models={t: [a for a, _ in ms] for t, ms in FINAL.items()},
    calibrators={"main": "calibrator_1_main.pkl", "fallback": "calibrator_2_fallback.pkl"},
    lanes={"fast_max_pd": LANE_FAST_PD, "high_risk_min_pd": LANE_HIGH_PD, "note": "calibrated PD cut-offs: <= fast = fast lane, >= high = strict review"},
    amount_suggestion={"risk_threshold_raw_monotone_pd": float(THR), "max_cut": 0.30, "step": 0.01,
                       "fold_models": [f"models/3_monotone__lightgbm__fold{k}.txt" for k in EVAL_FOLDS],
                       "noise_gate": "drop the suggestion if the PD drop is below 2 x SD across the fold models"},
    confidence_fold_models=[f"models/1_main__lightgbm__fold{k}.txt" for k in EVAL_FOLDS],
    sama={"cap_employee": SAMA_CAP_EMPLOYEE, "cap_retired": SAMA_CAP_RETIRED, "income_period": "monthly (assumption)"},
    eligibility={"min_age": MIN_AGE, "max_age_at_maturity": MAX_AGE_AT_MATURITY, "note": "example policy values, not SAMA numbers"},
    monotone_signs=dict(zip(FEATS, MONO)),
    arabic_labels=AR,
    cv_auc={f"{r.task}/{r.algo}": round(float(r.auc), 4) for r in summary.itertuples()},
    library_versions={"lightgbm": lgb.__version__, "xgboost": xgb.__version__, "catboost": __import__("catboost").__version__,
                      "sklearn": __import__("sklearn").__version__, "pandas": pd.__version__, "numpy": np.__version__},
)
TE.sample(300, random_state=SEED).sort_values("SK_ID_CURR").to_parquet(RUN / "final" / "demo_applicants.parquet", index=False)   # unlabeled applicants for the demo app
(RUN / "final" / "manifest.json").write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=1), encoding="utf-8")
(RUN / "final" / "serving.py").write_text('\"\"\"Scoring code for the website. Put this file next to manifest.json and the *.pkl files.\n    from serving import Scorer\n    s = Scorer(\"path/to/final\"); out = s.score(df)\ndf = one row per applicant with ALL model feature columns already built (same engineered features as the notebook).\nReturns a DataFrame: pd (calibrated probability of default), model_used, lane.\"\"\"\nimport json, pickle\nfrom pathlib import Path\nimport numpy as np, pandas as pd\n\n\nclass Scorer:\n    def __init__(self, folder):\n        self.f = Path(folder); self.man = json.loads((self.f / \"manifest.json\").read_text(encoding=\"utf-8\"))\n        self.models = {t: [(a, pickle.load(open(self.f / f\"{t}__{a}.pkl\", \"rb\"))) for a in algos] for t, algos in self.man[\"final_models\"].items()}\n        self.cal = {k: pickle.load(open(self.f / v, \"rb\")) for k, v in self.man[\"calibrators\"].items()}\n\n    def _prep(self, df, feats):\n        X = df.reindex(columns=feats).copy(); lv = self.man[\"categorical_levels\"]\n        for c in feats:\n            X[c] = pd.Categorical(X[c].astype(object).where(X[c].notna()), categories=lv[c]) if c in lv else X[c].astype(\"float32\")\n        return X\n\n    def _predict(self, task, X):\n        cat = [c for c in X.columns if c in self.man[\"categorical_levels\"]]; ps = []\n        for a, m in self.models[task]:\n            if a == \"lightgbm\":\n                ps.append(m.predict(X))\n            elif a == \"xgboost\":\n                ps.append(m.predict_proba(X.replace([np.inf, -np.inf], np.nan))[:, 1])\n            else:\n                Xc = X.copy()\n                for c in cat:\n                    Xc[c] = Xc[c].astype(object).where(Xc[c].notna(), \"NA\").astype(str)\n                ps.append(m.predict_proba(Xc.replace([np.inf, -np.inf], np.nan))[:, 1])\n        return np.mean(ps, axis=0)\n\n    def score(self, df):\n        feats = self.man[\"features\"]\n        ext_missing = df[[\"EXT_SOURCE_1\", \"EXT_SOURCE_2\", \"EXT_SOURCE_3\"]].isna().all(axis=1).values\n        p_main = self.cal[\"main\"].predict(self._predict(\"1_main\", self._prep(df, feats[\"main\"])))\n        p_fb = self.cal[\"fallback\"].predict(self._predict(\"2_fallback\", self._prep(df, feats[\"fallback\"])))\n        pdv = np.where(ext_missing, p_fb, p_main); L = self.man[\"lanes\"]\n        lane = np.where(pdv <= L[\"fast_max_pd\"], \"fast\", np.where(pdv >= L[\"high_risk_min_pd\"], \"high_risk\", \"normal\"))\n        return pd.DataFrame({\"pd\": pdv, \"model_used\": np.where(ext_missing, \"2_fallback\", \"1_main\"), \"lane\": lane}, index=df.index)\n', encoding="utf-8")
print("winners:", {"1_main": WIN_MAIN, "2_fallback": WIN_FB, "3_monotone": "lightgbm", "4_early_warning": WIN_EW})
print("saved in", RUN / "final", ":", sorted(p.name for p in (RUN / "final").iterdir()))
summary.round(4)

## Model selection summary

The delivered algorithm for each model with its cross-validated metrics (saved to `model_selection_summary.csv`).

In [ ]:
# MODEL SELECTION SUMMARY - one row per delivered model (numbers from the experiment tables)
def _sel_row(task, label, algo, tab):
    r = tab.set_index("algo").loc[algo]
    return dict(model=label, algorithm=algo, auc=round(float(r.auc), 4), pr_auc=round(float(r.pr_auc), 4), brier=round(float(r.brier), 4), ece=round(float(r.ece), 4),
                evaluated_rows=int(r.n_eval))
sel = pd.DataFrame([_sel_row("1_main", "1. Default risk", WIN_MAIN, tab_main), _sel_row("2_fallback", "2. Fallback (no external scores)", WIN_FB, tab_fb),
                    _sel_row("3_monotone", "3. Monotone (amount suggestion)", "lightgbm", tab_mono), _sel_row("4_early_warning", "4. Early warning", WIN_EW, tab_ew)])
sel["served_as"] = [", ".join(a for a, _ in FINAL[t]) for t in ("1_main", "2_fallback", "3_monotone", "4_early_warning")]
sel.to_csv(RUN / "model_selection_summary.csv", index=False)
sel

## Decision log

| # | Decision | Choice | Reason | Alternative considered |
|---|---|---|---|---|
| 1 | Prediction target | Probability of payment difficulty at application time | Matches the loan-officer decision | Early-warning only (kept as phase 2, Model 4) |
| 2 | Information used | Only data dated before the application | No leakage | Using the full history (would overstate performance) |
| 3 | Gender and age as inputs | Excluded; age is an eligibility rule | Non-discrimination; policy rules are explicit and auditable | Including them (small ranking gain, policy and fairness risk) |
| 4 | Validation | Fixed 5-fold stratified CV, same folds for all models | Honest, comparable results on imbalanced data | Single hold-out split (noisier) |
| 5 | Algorithms | Logistic regression, LightGBM, XGBoost, CatBoost, ensemble | Covers linear baseline and three boosting families | Neural networks (no clear advantage on tabular data of this kind, harder to explain) |
| 6 | Metrics | AUC, PR-AUC, Brier, ECE | Ranking, rare-class capture, probability accuracy | Accuracy (misleading with imbalance) |
| 7 | Model 1 | Ensemble of LightGBM, XGBoost and CatBoost | Highest cross-validated AUC and PR-AUC in the comparison; difference to the best single model is small, so this trades simplicity for accuracy by team decision | Single LightGBM (simpler, supports exact SHAP directly) |
| 8 | Calibration | Cross-fitted isotonic regression | Flexible, preserves ranking | Platt scaling (assumes a sigmoid shape) |
| 9 | Missing external scores | Separate fallback model without `EXT_SOURCE_*` | The main model depends heavily on them | Imputing the scores (hides the missingness) |
| 10 | Loan-amount suggestion | Monotone LightGBM so that risk never falls when amount or debt ratios rise; suggestion shown only if the risk change exceeds model noise | A non-monotone model could suggest "borrow more to be safer" | Unconstrained model; CatBoost monotone mode is CPU-only and impractically slow |
| 11 | Policy rules | SAMA salary-linked cap (1/3 employees, 25% retirees) and example age limits, outside the model | Regulation should be explicit, not learned | Learning the cap from data |
| 12 | Early warning | Separate model on monthly payment snapshots; folds grouped by client; compared with a simple benchmark | Different data and moment; avoids a client appearing in both train and test | Random row split (optimistic) |
| 13 | Final models | Retrain on 100% of rows with the tree count found in cross-validation | Uses all data; no hold-out needed for early stopping | Keeping the cross-validation models |
| 14 | Deployment | FastAPI service with a web UI; pickled models, calibrators and `manifest.json` | Recommended tool; simple to run and test | Notebook-only delivery |

## Conclusions

1. **The problem is solvable with the available data.** The tables above give the cross-validated performance of each model; Model 1 ranks applicants clearly better than chance and better than the linear baseline.
2. **Probabilities are usable as probabilities.** After isotonic calibration the expected calibration error is small, which allows the SAMA check, lanes and amount suggestions to use the probability value directly.
3. **Performance is not uniform across segments.** The segment audit shows which groups the model serves less well; this must be monitored in production and is a reason to keep gender and age out of the model.
4. **Model choice is a trade-off, not a large gain.** Differences between the boosting algorithms are small; the ensemble is adopted for Model 1 by team decision, while Models 3 and 4 use LightGBM.
5. **The fallback keeps the service usable** when external scores are missing, at lower accuracy.
6. **Policy rules stay explicit.** SAMA cap and eligibility are applied outside the learned models.

## Limitations

- The data is not Saudi data and the income period is undocumented; results must be re-validated on the lending bank's own portfolio before use.
- Age limits are example policy values, not SAMA figures; the bank must set them.
- The explanations show the model's own contributions; they are not causal statements.
- Hyperparameters were not tuned.
- The early-warning model is evaluated on one source (POS cash loans) and is a phase-2 component.

## Deployment hand-over

The final models are trained on 100% of the data and saved to `home_credit_runs/final/` together with:
- `manifest.json`: feature lists, category levels, lane thresholds, SAMA parameters, labels and library versions,
- `serving.py`: scoring code,
- `demo_applicants.parquet`: unlabeled applicants with full history for demonstrations.

The application repository loads this folder, serves it with **FastAPI**, and provides a web UI for manual entry and demo applicants. Library versions must match those in the manifest because the models are pickles.

## What we build on next

1. Connect **Model 3** (amount suggestion) and **Model 4** (early warning) to the application.
2. Replace example age limits and the income-period assumption with the bank's policy and data definitions.
3. **Recalibrate and re-validate** on the bank's portfolio; add a hyperparameter search.
4. Monitoring: score distribution, calibration drift and performance by segment; scheduled retraining.
5. Decide the fast-lane policy for age groups after reviewing the lane mix shown in the audit.
6. Add input validation and logging of every decision for audit.

### Reading the results

- Differences below about 0.002 AUC between algorithms are within fold-to-fold noise and should not drive a decision by themselves.
- Model 1 uses the ensemble of LightGBM, XGBoost and CatBoost; Models 3 and 4 use LightGBM.
- The deployment application needs the contents of `home_credit_runs/final/` (models, calibrators, `manifest.json`, `serving.py`) and `home_credit_runs/models/` (fold models used for confidence and the amount-suggestion noise gate).
- The scoring code expects the engineered feature columns to be present. They are produced by steps 2a to 2f.